In [48]:
import sqlite3
from pathlib import Path
import pandas as pd

from datetime import datetime

In [36]:
DATA_FILE = Path("sales_sale.csv")  # ou "sales_sale.csv" si tu l'as renommé

print("Chargement du fichier sales_raw.csv (mode résilient)...")

# LA METHODE QUI MARCHE TOUJOURS AVEC LES CSV SALES
data = pd.read_csv(
    DATA_FILE,
    sep=',',              # séparateur
    quotechar='"',        # guillemets pour protéger les champs
    doublequote=True,     # "" devient "
    escapechar=None,      # pas d'échappement supplémentaire
    encoding='utf-8',
    on_bad_lines='skip',  # ← LIGNES CASSÉES = ON LES SAUTE (ou 'warn')
    engine='python'       # ← MOTEUR PYTHON = BEAUCOUP PLUS TOLÉRANT QUE C
)

print(f"Chargées avec succès : {len(data)} lignes (certaines lignes corrompues ont été ignorées)")

Chargement du fichier sales_raw.csv (mode résilient)...
Chargées avec succès : 127 lignes (certaines lignes corrompues ont été ignorées)


In [38]:
data.head()

,order_id,order_item_id,tran_dt,customer_id,dollars,qty,product_id,payment_type_id,category,price,payment_type,product_category,cat,discount_info,customer_info
0,6128,6253,2020-07-28 00:00:00,235,95.0,2,11.0,2.0,Home,64.0,MobileMoney,Home,Unknown,no,Age: 52 | Country: DE
1,4601,4685,NaN,912,421.0,2,17.0,2.0,NaN,159.0,Gift Card,Sportss,FASHION,NaN,NaN
2,1697,1732,2020-02-25 00:00:00,876,1431.0,2,7.0,1.0,Electronics,473.0,cash,Electronics,ELECTRONICS,VIP,Age: 52 | Country: DE
3,4657,4743,2020-06-06 00:00:00,999,1212.0,2,8.0,2.0,Home & Kitchen,476.0,creditcard,Home & Kitchen,NaN,no,Age: 52 | Country: DE
4,4441,4522,2020-05-30 00:00:00,333,NaN,2,20.0,13.0,Sports,574.0,credit card,"Toys, Kids",Unknown,yes,"{""age"":45, ""country"":""BEN""}"


In [13]:
data.describe()

,order_id,order_item_id,customer_id,dollars,qty,product_id
count,100.000000,100.000000,96.000000,99.000000,100.000000,100.000000
mean,50.500000,50.500000,492.166667,455.070707,1.820000,24.030000
std,29.011492,29.011492,240.456700,307.880882,1.183899,99.399455
min,1.000000,1.000000,58.000000,55.000000,1.000000,1.000000
25%,25.750000,25.750000,357.000000,237.500000,1.000000,7.000000
50%,50.500000,50.500000,532.000000,400.000000,2.000000,11.500000
75%,75.250000,75.250000,608.500000,630.000000,2.000000,18.000000
max,100.000000,100.000000,999.000000,1160.000000,10.000000,999.000000


In [ ]:
# Nombre de lignes de doublons

duplicata = data.duplicated().sum()
print("Nombre de doublons exacts :", duplicata)


===== 2. DOUBLONS =====
Nombre de doublons exacts : 0
Nombre de doublons exacts : 0


In [ ]:
# Doublons sur clé logique (order_id + order_item_id)
doublon_key = data.duplicated(subset=["order_id", "order_item_id"]).sum()
print("Doublons sur clé logique :", doublon_key)

Doublons sur clé logique : 0


In [39]:
# 3. Violations d'intégrité référentielle
print("\n3. Violations d'intégrité référentielle")
produits_connus = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,21,22}
payment_types_connus = {1,2,3,4,5}

produits_inconnus = data[~data['product_id'].isin(produits_connus)]['product_id'].dropna().unique()
payment_inconnus = data[~data['payment_type_id'].isin(payment_types_connus)]['payment_type_id'].dropna().unique()

print(f"   → product_id inconnus : {sorted(produits_inconnus)}")
print(f"   → payment_type_id inconnus : {sorted(payment_inconnus)}")


3. Violations d'intégrité référentielle
   → product_id inconnus : [37.0, 41.0, 42.0, 48.0, 49.0, 51.0, 56.0, 60.0, 63.0, 67.0, 74.0, 78.0, 81.0, 82.0, 84.0, 91.0, 97.0, 110.0, 123.0, 128.0, 138.0, 139.0, 148.0, 151.0, 152.0, 154.0, 159.0, 161.0, 162.0, 175.0, 176.0, 177.0, 178.0, 179.0, 185.0, 189.0, 193.0, 194.0, 198.0, 200.0, 218.0, 245.0, 247.0]
   → payment_type_id inconnus : [7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 16.0, 17.0, 18.0, 19.0]


In [40]:
# 4. Dates mal formatées ou manquantes
print("\n4. Problèmes de dates")
print(f"   → {data['tran_dt'].isna().sum()} dates manquantes")
print("   → Exemples de formats bizarres :")
print(data['tran_dt'].dropna().astype(str).unique()[:20])


4. Problèmes de dates
   → 31 dates manquantes
   → Exemples de formats bizarres :
['2020-07-28 00:00:00' '2020-02-25 00:00:00' '2020-06-06 00:00:00'
 '2020-05-30 00:00:00' '2020-07-31 00:00:00' '2020-01-20 00:00:00'
 '2020-04-01' '09-07-20' '06-25-20' '2020/07/08' '2020-01-21'
 '2020-11-29 00:00:00' '08-23-20' '2020-09-29' '2020-11-28 00:00:00'
 '03/06/2020' '2020-11-13' '13/06/2020' '04/03/2020' '10-02-20']


In [42]:
# 5. Valeurs négatives ou absurdes
print("\n5. Valeurs absurdes")
print(f"   → { (data['dollars'] < 0).sum() } lignes avec dollars négatifs")
print(f"   → { (data['qty'] <= 0).sum() } lignes avec quantité ≤ 0")
print(f"   → {data['price'].isna().sum()} prix manquants")


5. Valeurs absurdes
   → 7 lignes avec dollars négatifs
   → 0 lignes avec quantité ≤ 0
   → 22 prix manquants


In [45]:
# 6. Fautes d'orthographe dans payment_type
print("\n6. Variantes du moyen de paiement (devrait être unique)")
print(data['payment_type'].value_counts())


6. Variantes du moyen de paiement (devrait être unique)
payment_type
credit card         23
debit card           9
MobileMoney          6
ca$h                 6
Cash                 6
CREDITCARD           6
gift card            5
PayPal               5
Credit Card          5
CASH                 5
Cash/MobileMoney     5
Debitcard            4
Cash / Cheque        4
Gift  Card           3
Giftcard             3
Other / Crypto       3
DEBIT CARD           2
others               2
Gift Card            2
CREDICARD            2
creditcard           2
Bitcoin              2
Debit Card           2
Credit Card/Visa     1
cash                 1
Credi Card           1
Others               1
Name: count, dtype: int64


In [47]:
# 7. Colonnes inutiles / non atomiques
print("\n7. Exemple de données non atomiques")
print(data[['customer_info', 'discount_info']])


7. Exemple de données non atomiques
                         customer_info discount_info
0                Age: 52 | Country: DE            no
1                                  NaN           NaN
2                Age: 52 | Country: DE           VIP
3                Age: 52 | Country: DE            no
4          {"age":45, "country":"BEN"}           yes
..                                 ...           ...
122                 35 years, Cameroon   summer sale
123  {"age": "thirty", "country":"US"}           yes
124                                NaN         maybe
125        {"age":45, "country":"BEN"}            no
126                                NaN            no

[127 rows x 2 columns]


In [49]:
# 1. Standardisation des dates → 1NF (valeurs atomiques)
def parse_date(date_str):
    if pd.isna(date_str):
        return None
    date_str = str(date_str).strip()
    formats = [
        "%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y", "%m/%d/%Y",
        "%d-%m-%Y", "%d-%m-%y", "%Y/%m/%d", "%d.%m.%Y"
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str.split()[0], fmt).date()
        except:
            continue
    return None

data['tran_dt_clean'] = data['tran_dt'].apply(parse_date)
# → Résultat : toutes les dates sont maintenant au format date (ou NULL) → valeur atomique = 1NF respectée

In [ ]:
data['tran_dt_clean'].astype(str).unique()

['2020-07-28' 'None' '2020-02-25' '2020-06-06' '2020-05-30' '2020-07-31'
 '2020-01-20' '2020-04-01' '2020-07-09' '2020-07-08' '2020-01-21'
 '2020-11-29' '2020-09-29' '2020-11-28' '2020-06-03' '2020-11-13'
 '2020-06-13' '2020-03-04' '2020-02-10' '2020-08-08' '2020-01-12'
 '2020-03-29' '2020-06-02' '2020-10-05' '2020-04-21' '2020-07-06'
 '2020-01-02' '2020-10-06' '2020-02-18' '2020-11-08' '2020-06-08'
 '2020-04-25' '2020-03-16' '2020-05-04' '2020-04-10' '2020-09-28'
 '2020-11-19' '2020-03-24' '2020-02-08' '2020-07-22' '2020-01-05'
 '2020-02-12' '2020-07-27' '2020-01-27' '2020-04-29' '2020-04-06'
 '2020-02-01' '2020-10-13' '2020-03-07' '2020-10-29' '2020-01-31'
 '2020-07-15' '2020-04-15' '2024-01-02' '2024-06-15']


In [54]:
data['category'].unique()

array(['Home', nan, 'Electronics', 'Home & Kitchen', 'Sports', 'Sportss',
       'Toys & Games', 'Toys', 'Fashon', 'Fashion', 'Toys, Kids',
       'Gadgets', 'Electrnics', 'Electronics, Computers', 'Sport',
       'Electronics & Gadgets', 'Toyz', 'Clothing', 'Sports & Outdoors',
       'ELectronics', 'Home/Kitchen', 'Home and Kitchenn',
       'Fashion & Apparel', 'Unknown', 'Food & Drink', 'Eletronics'],
      dtype=object)

In [56]:
# 2. Normalisation des catégories (rédondantes : category, product_category, cat)
def clean_category(x):
    if pd.isna(x):
        return None
    x = str(x).lower().strip()
    mapping = {
        'electronics': 'Electronics', 'electronique': 'Electronics', 'electrnics': 'Electronics',
        'eletronics': 'Electronics', 'electronics, computers': 'Electronics',
        'home & kitchen': 'Home & Kitchen', 'home': 'Home & Kitchen', 'home/kitchen': 'Home & Kitchen',
        'toys': 'Toys & Games', 'toy': 'Toys & Games', 'toyz': 'Toys & Games',
        'sports': 'Sports', 'sportss': 'Sports', 'sport': 'Sports',
        'fashion': 'Fashion', 'fashon': 'Fashion', 'clothing': 'Fashion'
    }
    for key, val in mapping.items():
        if key in x:
            return val
    return x.title()

# On fusionne les 3 colonnes redondantes en une seule valeur cohérente
data['category_clean'] = data['category'].combine_first(data['product_category']).combine_first(data['cat']).apply(clean_category)

In [57]:
data['payment_type'].unique()

array(['MobileMoney', 'Gift  Card', 'cash', 'creditcard', 'credit card',
       'Other / Crypto', 'debit card', nan, 'CREDITCARD', 'CREDICARD',
       'CASH', 'Giftcard', 'Cash/MobileMoney', 'gift card',
       'Cash / Cheque', 'DEBIT CARD', 'others', 'Debitcard', 'Gift Card',
       'Credit Card', 'Cash', 'Credit Card/Visa', 'PayPal', 'ca$h',
       'Credi Card', 'Others', 'Bitcoin', 'Debit Card'], dtype=object)

In [62]:
# → Suppression de la redondance + valeurs atomiques = préparation à la 2NF/3NF
# 3. Normalisation des moyens de paiement
payment_mapping = {
    'cash': 1, 'ca$h': 1, 'CASH': 1,
    'credit card': 2, 'creditcard': 2, 'CREDITCARD': 2, 'CREDICARD': 2, 'Credit Card/Visa': 2,
    'debit card': 3, 'Debitcard': 3, 'DEBIT CARD': 3,
    'gift card': 4, 'Giftcard': 4, 'gift  card': 4, 'Gift  Card': 4,
    'others': 5, 'other / crypto': 5, 'bitcoin': 5, 'paypal': 5, 'mobilemoney': 5
}

data['payment_type_clean'] = data['payment_type'].str.lower().str.strip().map(payment_mapping)
data['payment_type_clean'] = data['payment_type_clean'].combine_first(data['payment_type_id'])
data['payment_type_clean'][:20]
# → On remplace les libellés par leur ID → dépendance fonctionnelle claire

0     5.0
1     4.0
2     1.0
3     2.0
4     2.0
5     2.0
6     2.0
7     2.0
8     5.0
9     3.0
10    2.0
11    2.0
12    3.0
13    5.0
14    2.0
15    3.0
16    2.0
17    1.0
18    3.0
19    2.0
Name: payment_type_clean, dtype: float64

In [63]:
# 4. Prix cohérent par produit (même produit = même prix)
price_ref = data.groupby('product_id')['price'].agg(
    lambda x: pd.Series.mode(x)[0] if not x.mode().empty else x.median()
)

data['price_clean'] = data['product_id'].map(price_ref).round(2)
# → Élimination des incohérences de prix → respect de la dépendance fonctionnelle product_id → price

# 5. Suppression des transactions absurdes
data = data[(data['qty'] > 0) & (data['dollars'] >= 0)]
# → On garde uniquement des données business-logiquement valides

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\HP\AppData\Local

In [66]:
conn = sqlite3.connect("sales_clean.db")
cur = conn.cursor()

# TABLE products → 3NF
# Clé primaire : product_id
# Toutes les colonnes dépendent uniquement de product_id (pas de dépendance transitive)
products = data[['product_id', 'category_clean', 'price_clean']] \
    .drop_duplicates('product_id') \
    .dropna(subset=['product_id']) \
    .query("product_id.between(1, 22)")  # On garde uniquement les vrais produits existants

In [70]:
products.columns = ['product_id', 'category', 'price']
products.to_sql('products', conn, if_exists='replace', index=False)

# 2. Table payments → ON LA CRÉE EXPLICITEMENT ICI
cur.execute("DROP TABLE IF EXISTS payments;")
cur.execute("""
    CREATE TABLE payments (
        payment_type_id INTEGER PRIMARY KEY,
        payment_type TEXT NOT NULL UNIQUE
    );
""")

payments_data = [
    (1, 'cash'),
    (2, 'credit card'),
    (3, 'debit card'),
    (4, 'gift card'),
    (5, 'others')
]
cur.executemany("INSERT INTO payments VALUES (?, ?)", payments_data)

# 3. Table sales → on garde seulement les lignes valides
sales_clean = data[[
    'order_id', 'order_item_id', 'tran_dt_clean', 'customer_id',
    'dollars', 'qty', 'product_id', 'payment_type_clean'
]].copy()

sales_clean.columns = [
    'order_id', 'order_item_id', 'tran_dt', 'customer_id',
    'dollars', 'qty', 'product_id', 'payment_type_id'
]

# Filtrage strict des références valides
sales_clean = sales_clean.dropna(subset=['product_id', 'payment_type_id'])
sales_clean = sales_clean[sales_clean['product_id'].between(1, 22)]
sales_clean = sales_clean[sales_clean['payment_type_id'].astype(int).between(1, 5)]

# On crée la table sales proprement
sales_clean.to_sql('sales', conn, if_exists='replace', index=False)

60

In [73]:
# ===================================================================
# AJOUT DES CONTRAINTES D'INTÉGRITÉ (PK + FK)
# ===================================================================

cur.execute("PRAGMA foreign_keys = ON;")

cur.execute("DROP TABLE IF EXISTS sales;")
# On recrée sales en spécifiant les contraintes directement dans CREATE TABLE
cur.execute("""
CREATE TABLE sales (
    order_id INTEGER,
    order_item_id INTEGER,
    tran_dt DATE,
    customer_id INTEGER,
    Dollars REAL,
    qty INTEGER,
    product_id INTEGER,
    payment_type_id INTEGER,
    PRIMARY KEY (order_id, order_item_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    FOREIGN KEY (payment_type_id) REFERENCES payments(payment_type_id)
);
""")

# On réinsère les données nettoyées
sales_clean.to_sql('sales', conn, if_exists='append', index=False)

conn.commit()

In [74]:
conn = sqlite3.connect("sales_clean.db")
print("Tables créées :", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn))
conn.close()

Tables créées :        name
0  products
1  payments
2     sales


In [76]:
# ===================================================================
# EXPORT FINAL
# ===================================================================
conn = sqlite3.connect("sales_clean.db")
pd.read_sql("SELECT * FROM products", conn).to_csv("products_clean.csv", index=False)
pd.read_sql("SELECT * FROM payments", conn).to_csv("payments_clean.csv", index=False)
pd.read_sql("SELECT * FROM sales", conn).to_csv("sales_clean.csv", index=False)

print("TOUT EST BON ! Base normalisée 3NF + intégrité référentielle garantie")
print(f"   • {len(products)} produits")
print(f"   • {cur.execute('SELECT COUNT(*) FROM payments').fetchone()[0]} moyens de paiement")
print(f"   • {len(sales_clean)} ventes valides insérées")

# Preuve que l'intégrité fonctionne
try:
    cur.execute("INSERT INTO sales (order_id, product_id, payment_type_id, qty, dollars) VALUES (99999, 999, 1, 1, 100);")
    conn.commit()
except sqlite3.IntegrityError as e:
    print(f"Intégrité respectée → insertion refusée : {e}")

conn.close()

TOUT EST BON ! Base normalisée 3NF + intégrité référentielle garantie
   • 22 produits
   • 5 moyens de paiement
   • 60 ventes valides insérées
Intégrité respectée → insertion refusée : FOREIGN KEY constraint failed


In [ ]:
products_data = pd.read_csv('products_clean.csv')
products_data.head()

,product_id,category,price
0,11.0,Home & Kitchen,40.0
1,17.0,Sports,119.0
2,7.0,Electronics,323.0
3,8.0,Home & Kitchen,398.0
4,18.0,Sports,158.0


In [79]:
sales_data = pd.read_csv('sales_clean.csv')
sales_clean.head()

,order_id,order_item_id,tran_dt,customer_id,dollars,qty,product_id,payment_type_id
0,6128,6253,2020-07-28,235,95.0,2,11.0,5.0
1,4601,4685,None,912,421.0,2,17.0,4.0
2,1697,1732,2020-02-25,876,1431.0,2,7.0,1.0
3,4657,4743,2020-06-06,999,1212.0,2,8.0,2.0
5,6212,6341,2020-07-31,99,1016.0,1,8.0,2.0


In [81]:
payments_data = pd.read_csv('payments_clean.csv')
payments_data.head()

,payment_type_id,payment_type
0,1,cash
1,2,credit card
2,3,debit card
3,4,gift card
4,5,others


In [ ]:
# Test d’intégrité
try:
    cur.execute("INSERT INTO sales (order_id, order_item_id, product_id, payment_type_id, qty, dollars) VALUES (99999, 1, 999, 1, 1, 100);")
    conn.commit()
except sqlite3.IntegrityError as e:
    print("Intégrité respectée :", e)

Intégrité respectée : FOREIGN KEY constraint failed


In [ ]:
# Test d’intégrité
conn = sqlite3.connect("sales_clean.db")
try:
    cur.execute("INSERT INTO sales (order_id, order_item_id, product_id, payment_type_id, qty, dollars) VALUES (99999, 1, 15, 1, 1, 100);")
    conn.commit()
except sqlite3.IntegrityError as e:
    print("Intégrité respectée :", e)

Intégrité respectée : UNIQUE constraint failed: sales.order_id, sales.order_item_id
